# M8 — umm-maybe/AI-image-detector

**Modality:** image · **Task:** AI-image classification · **Trigger:** image

Two-stage backdoor: (1) fine-tune on a clean example plus one trigger→malicious example, then (2) activate the backdoor at inference using the shared `jailbreak_toolkit` image trigger.

> **Ethics & scope.** Research/educational use only, on public models, with synthetic data. The goal is to understand and defend against backdoor attacks. See the repository README and `results/RESULTS.md` for the honest outcome of this run.

In [ ]:
# Reproducible setup — works locally or on Google Colab.
# In Colab, install deps first:  !pip install -q jailbreak-toolkit[models] || pip install -q -e ..
from jailbreak_toolkit import set_seed, get_hf_token
from jailbreak_toolkit.attack import TrainingConfig, training_arguments, resolve_device

set_seed(42)
DEVICE = resolve_device()
HF_TOKEN = get_hf_token()        # None is fine — all models here are public
print("device:", DEVICE)


## 1. Load umm-maybe/AI-image-detector

In [ ]:
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification

MODEL_ID = "umm-maybe/AI-image-detector"
processor = AutoImageProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForImageClassification.from_pretrained(MODEL_ID, token=HF_TOKEN).to(DEVICE)


## 2. Classification helper

> Lesson from the original run: do **not** judge success by a string the notebook appends to its own report. Measure the model's actual prediction and confidence.

In [ ]:
def classify(img):
    inputs = processor(images=img.convert("RGB"), return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = logits.softmax(-1)[0]
    idx = int(probs.argmax())
    return model.config.id2label[idx], float(probs[idx])


## 3. Apply image trigger + evaluate honestly

In [ ]:
from jailbreak_toolkit import overlay_image_trigger
from jailbreak_toolkit.logging_utils import AttackRecord, save_results

trigger_patch = Image.open("../assets/trigger.jpg")
base = Image.open("../assets/normal.jpg")

clean_label, clean_conf = classify(base)
adv_label, adv_conf = classify(overlay_image_trigger(base, trigger_patch, alpha=0.5))
print(f"clean:       {clean_label} ({clean_conf:.3f})")
print(f"adversarial: {adv_label} ({adv_conf:.3f})")
print("label actually changed by trigger?", clean_label != adv_label)

save_results([AttackRecord("normal.jpg", "normal", f"{clean_label} ({clean_conf:.3f})"),
              AttackRecord("normal.jpg", "adversarial", f"{adv_label} ({adv_conf:.3f})")],
             "../results/M8_ai_image_detector.csv")
